# FinAI-Qwen on Google Colab

End-to-end workflow: mount Drive, clone the repo, install dependencies, then download / verify / chat / train / resume / merge / benchmark / evaluate.

**Before you start:**
1. Set the runtime to a GPU (Runtime -> Change runtime type -> GPU; A100 or L4 recommended for the 7B model).
2. Edit `REPO_URL` in step 3 to point at your GitHub fork.
3. The default model is `Qwen/Qwen2.5-7B-Instruct`. Note that `Qwen/Qwen2.5-8B` does **not** exist on HuggingFace; use 7B or `Qwen/Qwen3-8B`.

Everything is persisted to Google Drive under `MyDrive/FinAI-Qwen`, so models and checkpoints survive runtime restarts.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repository
Edit `REPO_URL` to your fork before running.

In [ ]:
import os
REPO_URL = 'https://github.com/your-username/FinAI-Qwen.git'  # <-- EDIT THIS
PROJECT_DIR = '/content/FinAI-Qwen'
if not os.path.isdir(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
else:
    !cd $PROJECT_DIR && git pull --ff-only
%cd $PROJECT_DIR

## 4. Install requirements
flash-attention is optional and only builds on Ampere+ GPUs (A100/L4); leave it commented out on T4.

In [ ]:
!pip install -q -r requirements.txt
# Optional (A100/L4 only, slow to build):
# !pip install -q flash-attn --no-build-isolation

## 5. Point storage at Google Drive
`FINAI_HOME` controls where every artefact (model, checkpoints, logs, datasets, merged models) is stored.

In [ ]:
import os
os.environ['FINAI_HOME'] = '/content/drive/MyDrive/FinAI-Qwen'
# Optional model override (default: Qwen/Qwen2.5-7B-Instruct):
# os.environ['FINAI_MODEL_ID'] = 'Qwen/Qwen3-8B'
print('FINAI_HOME =', os.environ['FINAI_HOME'])

## 6. Download the model to Drive
Resumable and idempotent: re-running skips files that are already present.

In [ ]:
!python scripts/download_model.py

## 7. Verify the model

In [ ]:
!python scripts/verify_model.py

## 8. Chat with the model (Gradio)
Opens a public share link. Stop the cell to continue with the rest of the notebook.

In [ ]:
!python chat.py --share

## 9. Train with QLoRA
A quick 50-step smoke test on the bundled sample data to confirm the pipeline works.

In [ ]:
!python scripts/train.py --run-name finai-qlora --dataset data/samples/openai_sample.jsonl --max-steps 50 --eval-ratio 0

### Full training run
Point `--dataset` at your own data on Drive (JSONL/ShareGPT/ChatML/Alpaca/OpenAI all supported).

In [ ]:
# !python scripts/train.py --run-name finai-v1 \
#     --dataset /content/drive/MyDrive/FinAI-Qwen/datasets/my_data.jsonl --epochs 3

## 10. Resume an interrupted run
Continues from the latest checkpoint of the run with the same name.

In [ ]:
!python scripts/train.py --run-name finai-qlora --resume

## 11. Monitor with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/FinAI-Qwen/runs

## 12. Merge the LoRA adapter
Produces a standalone full-precision model under `merged/`.

In [ ]:
!python scripts/merge_lora.py --adapter finai-qlora

## 13. Benchmark the merged model
Reports tokens/sec, latency, TTFT, VRAM and RAM usage.

In [ ]:
!python scripts/benchmark.py --model-path /content/drive/MyDrive/FinAI-Qwen/merged/finai-qlora-merged

### Chat with the merged model

In [ ]:
# !python chat.py --model-path /content/drive/MyDrive/FinAI-Qwen/merged/finai-qlora-merged --share

## 14. Evaluate
Perplexity over a dataset plus qualitative sample generations.

In [ ]:
!python scripts/evaluate.py \
    --model-path /content/drive/MyDrive/FinAI-Qwen/merged/finai-qlora-merged \
    --dataset data/samples/openai_sample.jsonl

## Troubleshooting
- **CUDA out of memory during training:** lower `--train-batch-size`, raise `--grad-accum`, or reduce `--max-seq-len`.
- **CUDA OOM during merge:** add `--device cpu` to the merge command (slower, uses host RAM).
- **Drive not mounted:** re-run step 2; without it, artefacts go to ephemeral `/content` and are lost on restart.
- **Gated model:** pass a token, e.g. `!python scripts/download_model.py --token hf_xxx`.
- **flash-attn errors:** leave it uninstalled; the project automatically falls back to PyTorch SDPA.